In [7]:
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize
import numpy as np

# Download all required NLTK data
print("Downloading required NLTK data...")
nltk.download('punkt')
nltk.download('punkt_tab')

def split_speech_into_sentences(df):
    # Create a list to store the new rows
    new_rows = []
    
    # Process each row in the original dataframe
    for idx, row in df.iterrows():
        speech_text = row['speech_text']
        
        # Skip if speech_text is NaN
        if pd.isna(speech_text):
            continue
            
        # Get sentences using NLTK
        sentences = sent_tokenize(speech_text)
        
        # Create new row for each sentence
        for sentence in sentences:
            # Create a copy of the original row
            new_row = row.copy()
            # Replace the speech_text with the single sentence
            new_row['speech_text'] = sentence.strip()
            new_rows.append(new_row)
    
    # Create new dataframe from the rows
    return pd.DataFrame(new_rows)

# Test on a small sample
def test_sentence_splitting():
    # Create sample data
    sample_data = {
        'speaker_id': [1, 2],
        'speech_text': [
            "This is a test speech. It has multiple sentences! What about Mr. Smith? He's here.",
            "Another speech with Dr. Jones, Ph.D. They work at the U.S. Dept. of Defense."
        ],
        'date': ['2023-01-01', '2023-01-01'],
        'party': ['Party A', 'Party B']
    }
    
    df = pd.DataFrame(sample_data)
    
    # Process the sample
    result_df = split_speech_into_sentences(df)
    
    # Print results
    print("Original number of rows:", len(df))
    print("New number of rows:", len(result_df))
    print("\nSample of split sentences:")
    print(result_df[['speaker_id', 'speech_text']].head())
    
    return result_df


def process_real_data_sample():
    # Read your CSV file - replace with your actual path
    df = pd.read_csv('speeches_2014-2024_speakers_parties_fixed.csv')
    
    # Print columns to verify
    print("Columns in dataset:", df.columns.tolist())
    
    # Take a small sample (e.g., first 100 rows)
    sample_df = df.head(100)
    
    # Process the sample
    result_df = split_speech_into_sentences(sample_df)
    
    # Print summary statistics
    print("\nOriginal number of rows:", len(sample_df))
    print("New number of rows:", len(result_df))
    print("\nSample of split sentences (showing all columns):")
    print(result_df.head())
    
    # Save the results to a new CSV
    result_df.to_csv('speeches_sentences_sample.csv', index=False)
    
    # Print column verification
    print("\nVerifying all columns were preserved:")
    print("Original columns:", sample_df.columns.tolist())
    print("New columns:", result_df.columns.tolist())
    
    return result_df

# Run with your actual data
real_data_df = process_real_data_sample()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\anouk\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\anouk\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Columns in dataset: ['file_id', 'speaker_name', 'speaker_party', 'speech_text', 'jaar', 'date', 'kamer', 'category', 'title', 'document_number', 'url', 'meta_url', 'vergadernummer', 'standardized_name']

Original number of rows: 100
New number of rows: 581

Sample of split sentences (showing all columns):
               file_id   speaker_name speaker_party  \
0  h-tk-20152016-52-16  De voorzitter           NaN   
0  h-tk-20152016-52-16  De voorzitter           NaN   
0  h-tk-20152016-52-16  De voorzitter           NaN   
0  h-tk-20152016-52-16  De voorzitter           NaN   
1  h-tk-20152016-52-16  De voorzitter           NaN   

                                         speech_text       jaar        date  \
0  De motie-Gesthuizen/Kerstens (\nDe Kamer,\ngeh...  2015-2016  2016-02-09   
0  Naar mij blijkt, wordt de indiening ervan vold...  2015-2016  2016-02-09   
0                                     Zij krijgt nr.  2015-2016  2016-02-09   
0  Ik stel vast dat wij hier nu over kunnen st

In [12]:
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize
import numpy as np
import re

# Download required NLTK data
print("Downloading required NLTK data...")
nltk.download('punkt')

def clean_parliamentary_text(text):
    if pd.isna(text):
        return ""
        
    # Handle motion numbers (prevent splitting on nr. and numbers)
    text = re.sub(r'(\([\w\s]+,\s*nr\.\s*\d+[^)]*\))', lambda m: m.group().replace('.', '@'), text)
    
    # Handle common parliamentary formatting
    text = text.replace('\n', ' ')  # Replace newlines with spaces
    text = re.sub(r'(?<=\w);(?=\s)', '. ', text)  # Convert semicolons to periods if between words
    text = re.sub(r'\s+', ' ', text)  # Clean multiple spaces
    
    # Restore periods in motion numbers
    text = text.replace('@', '.')
    
    return text.strip()

def split_speech_into_sentences(df):
    new_rows = []
    
    for idx, row in df.iterrows():
        speech_text = clean_parliamentary_text(row['speech_text'])
        
        if not speech_text:
            continue
            
        # Get sentences using NLTK
        sentences = sent_tokenize(speech_text)
        
        # Create new row for each sentence
        for i, sentence in enumerate(sentences, 1):
            sentence = sentence.strip()
            if len(sentence) > 0:
                new_row = row.copy()
                new_row['speech_text'] = sentence
                new_row['sentence_number'] = i  # Track sentence position
                new_row['speech_id'] = f"{row['file_id']}_{i}"  # Create unique identifier
                new_rows.append(new_row)
    
    return pd.DataFrame(new_rows)

def analyze_consecutive_speeches(start_index=0, n=10):
    """Analyze n consecutive speeches starting from start_index"""
    # Read original data
    df = pd.read_csv('speeches_2014-2024_speakers_parties_fixed.csv')
    
    # Take n consecutive speeches
    sample_df = df.iloc[start_index:start_index + n]
    
    # Process sample
    result_df = split_speech_into_sentences(sample_df)
    
    # Print analysis
    print(f"\nAnalyzing {n} consecutive speeches starting from index {start_index}:")
    print(f"Original speeches: {len(sample_df)}")
    print(f"Total sentences: {len(result_df)}")
    print(f"Average sentences per speech: {len(result_df)/len(sample_df):.1f}")
    
    # Print detailed comparison
    print("\nDetailed comparison (in chronological order):")
    for idx, row in sample_df.iterrows():
        original_text = row['speech_text']
        sentences = result_df[result_df['file_id'] == row['file_id']]['speech_text'].tolist()
        
        print(f"\nSpeech {row['file_id']}:")
        print(f"Speaker: {row['speaker_name']}")
        print(f"Original length: {len(str(original_text))} chars")
        print(f"Sentences found: {len(sentences)}")
        print("First 2 sentences:")
        for i, sent in enumerate(sentences[:2], 1):
            print(f"{i}. {sent}")
        print("-" * 80)
    
    # Save results
    result_df.to_csv('speeches_sentences_sample_improved.csv', index=False)
    return result_df

# Run analysis on 10 consecutive speeches
sample_results = analyze_consecutive_speeches(start_index=0, n=10)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\anouk\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!



Analyzing 10 consecutive speeches starting from index 0:
Original speeches: 10
Total sentences: 31
Average sentences per speech: 3.1

Detailed comparison (in chronological order):

Speech h-tk-20152016-52-16:
Speaker: De voorzitter
Original length: 975 chars
Sentences found: 8
First 2 sentences:
1. De motie-Gesthuizen/Kerstens ( De Kamer, gehoord de beraadslaging, constaterende dat bij het Europees Octrooibureau in Rijswijk en München al geruime tijd sprake is van gerichte aanvallen op vakbondsleden en een zeer hoge werkdruk.
2. voorts constaterende dat het EOB als internationale organisatie niet onder het nationale arbeidsrecht valt.
--------------------------------------------------------------------------------

Speech h-tk-20152016-52-16:
Speaker: De voorzitter
Original length: 275 chars
Sentences found: 8
First 2 sentences:
1. De motie-Gesthuizen/Kerstens ( De Kamer, gehoord de beraadslaging, constaterende dat bij het Europees Octrooibureau in Rijswijk en München al geruime tijd 

In [ ]:
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize
import re

# Download required NLTK data
print("Downloading required NLTK data...")
nltk.download('punkt')

def custom_sentence_tokenize(text):
    if pd.isna(text):
        return []
    
    # Find all parenthetical expressions and replace them temporarily
    parentheses = re.finditer(r'\([^)]*\)', text)
    replacements = {}
    
    for i, match in enumerate(parentheses):
        placeholder = f"PARENTH_{i}_"
        original = match.group()
        replacements[placeholder] = original
        text = text.replace(original, placeholder)
    
    # Split into sentences
    sentences = sent_tokenize(text)
    
    # Restore parenthetical expressions
    restored_sentences = []
    for sentence in sentences:
        for placeholder, original in replacements.items():
            sentence = sentence.replace(placeholder, original)
        restored_sentences.append(sentence.strip())
    
    return restored_sentences

def split_speech_into_sentences(df):
    new_rows = []
    
    for idx, row in df.iterrows():
        sentences = custom_sentence_tokenize(row['speech_text'])
        
        # Create new row for each sentence
        for i, sentence in enumerate(sentences, 1):
            if len(sentence) > 0:
                new_row = row.copy()
                new_row['speech_text'] = sentence
                new_row['sentence_number'] = i
                new_row['speech_id'] = f"{row['file_id']}_{i}"
                new_rows.append(new_row)
    
    return pd.DataFrame(new_rows)

# Quick test
test_text = "In stemming komt de motie-Lucas (31524, nr. 283). De volgende motie."
print("Test case:")
print("Input:", test_text)
print("\nOutput sentences:")
for i, sent in enumerate(custom_sentence_tokenize(test_text), 1):
    print(f"{i}. {sent}")

# Main analysis function
def analyze_consecutive_speeches(start_index=0, n=10):
    # Read original data
    df = pd.read_csv('speeches_2014-2024_speakers_parties_fixed.csv')
    
    # Take n consecutive speeches
    sample_df = df.iloc[start_index:start_index + n]
    
    # Process sample
    result_df = split_speech_into_sentences(sample_df)
    
    # Print analysis
    print(f"\nAnalyzing {n} consecutive speeches:")
    print(f"Original speeches: {len(sample_df)}")
    print(f"Total sentences: {len(result_df)}")
    
    # Save results
    result_df.to_csv('speeches_sentences_sample_improved2.csv', index=False)
    return result_df

# Run analysis
sample_results = analyze_consecutive_speeches(start_index=0, n=10)

Test case:
Input: In stemming komt de motie-Lucas (31524, nr. 283). De volgende motie.

Output sentences:
1. In stemming komt de motie-Lucas (31524, nr. 283).
2. De volgende motie.


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\anouk\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!



Analyzing 10 consecutive speeches:
Original speeches: 10
Total sentences: 27


In [1]:
pip install spacy

   ---------------------------------------- 0.0/11.8 MB ? eta -:--:--
   --------- ------------------------------ 2.9/11.8 MB 14.0 MB/s eta 0:00:01
   ---------------------- ----------------- 6.6/11.8 MB 16.1 MB/s eta 0:00:01
   ----------------------------------- ---- 10.5/11.8 MB 16.4 MB/s eta 0:00:01
   ---------------------------------------- 11.8/11.8 MB 14.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/632.6 kB ? eta -:--:--
   --------------------------------------- 632.6/632.6 kB 12.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 12.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/6.3 MB ? eta -:--:--
   ----------------------- ---------------- 3.7/6.3 MB 18.1 MB/s eta 0:00:01
   ---------------------------------------- 6.3/6.3 MB 16.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/5.4 MB ? eta -:--:--
   ------------------------

In [6]:
pip install googletrans 


  Attempting uninstall: httpx
    Found existing installation: httpx 0.27.0
    Uninstalling httpx-0.27.0:
      Successfully uninstalled httpx-0.27.0


In [2]:
pip install deep-translator

Note: you may need to restart the kernel to use updated packages.


In [7]:
import pandas as pd
from googletrans import Translator
from deep_translator import GoogleTranslator


# Load CSV file
input_file = "speeches_sentences_sample_improved2.csv"  # Change to your actual file
output_file = "translated_sample_data.csv"

# Read CSV into DataFrame
df = pd.read_csv(input_file)

# Initialize Google Translator
translator = Translator()

# Function to translate text
def translate_text(text, dest_lang="en"):
    if pd.isna(text) or text.strip() == "":
        return text
    try:
        return GoogleTranslator(source="auto", target=dest_lang).translate(text)
    except Exception as e:
        print(f"Error translating: {text} -> {e}")
        return text

# Translate the 'speech_text' column
df["speech_text"] = df["speech_text"].apply(lambda x: translate_text(x, "en"))

# Save translated data
df.to_csv(output_file, index=False)

print(f"Translation completed! Saved as {output_file}")


Translation completed! Saved as translated_sample_data.csv


In [5]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------- ----------------------------- 3.4/12.8 MB 18.3 MB/s eta 0:00:01
     ---------------------- ----------------- 7.1/12.8 MB 17.4 MB/s eta 0:00:01
     ------------------------------ -------- 10.0/12.8 MB 16.8 MB/s eta 0:00:01
     --------------------------------------- 12.8/12.8 MB 15.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [8]:
import pandas as pd
import spacy

# Load the translated data
input_file = "translated_sample_data.csv"
df = pd.read_csv(input_file)

# Load English spaCy model
print("Loading English spaCy model...")
nlp_eng = spacy.load("en_core_web_sm")

# Add a POS-tagged version of each sentence
print("Adding POS tags...")
df["pos_tags"] = df["speech_text"].apply(
    lambda x: " ".join([f"{token.text}_{token.pos_}" for token in nlp_eng(str(x))])
)

# Save the updated DataFrame with POS tags
output_file = "translated_sample_data_with_pos.csv"
df.to_csv(output_file, index=False)

print(f"POS tagging completed! Saved as {output_file}")

# Optional: Show a sample of the result
print("\nSample of POS-tagged sentences:")
print(df[["speech_text", "pos_tags"]].head())

Loading English spaCy model...
Adding POS tags...
POS tagging completed! Saved as translated_sample_data_with_pos.csv

Sample of POS-tagged sentences:
                                         speech_text  \
0  The motion teeth/kerstens (\nThe room,\nheard ...   
1  It turns out that the submission of it is suff...   
2                                       She gets no.   
3               I note that we can now vote on this.   
4  I note that the members of the SP, the PvdD, P...   

                                            pos_tags  
0  The_DET motion_NOUN teeth_NOUN /_SYM kerstens_...  
1  It_PRON turns_VERB out_ADP that_SCONJ the_DET ...  
2                 She_PRON gets_VERB no_INTJ ._PUNCT  
3  I_PRON note_VERB that_SCONJ we_PRON can_AUX no...  
4  I_PRON note_VERB that_SCONJ the_DET members_NO...  
